Installing dependencies

In [ ]:
!pip install -q -U \
    transformers \
    accelerate \
    bitsandbytes \
    fastapi \
    "uvicorn[standard]" \
    python-multipart \
    soundfile \
    nest_asyncio \
    huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.1 which is incompatible.


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


Load secrets

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login


HF_TOKEN = userdata.get("HF_TOKEN")
GATEWAY_KEY = userdata.get("GATEWAY_KEY")

login(token=HF_TOKEN)

print("Hugging Face login successful.")
print("CUDA available:", torch.cuda.is_available())

Hugging Face login successful.
CUDA available: True


##Load N-ATLas LLM

using 4-bit quantitization to reduce GPU memory pressure


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

LLM_ID = "NCAIR1/N-ATLaS"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    LLM_ID,
    token=HF_TOKEN
)

llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto"
)

print("N-ATLaS loaded.")

config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

N-ATLaS loaded.


Testing N-ATLaS

In [ ]:
from datetime import datetime

messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful academic assistant "
            "for Nigerian students. "
            "Give clear, accurate and concise answers."
        )
    },
    {
        "role": "user",
        "content": (
            "Explain the difference between stress "
            "and strain in simple terms."
        )
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
    date_string=datetime.now().strftime("%d %b %Y")
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    add_special_tokens=False
)

inputs = {
    k: v.to(llm.device)
    for k, v in inputs.items()
}

with torch.no_grad():

    outputs = llm.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.1
    )

generated = outputs[0][
    inputs["input_ids"].shape[1]:
]

answer = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print(answer)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Sure! Here's an explanation of the difference between stress and strain in simple terms:

- Stress is like the pressure applied to something (like a rubber band) that causes it to bend or deform. Think of it as the external force being put on an object.

- Strain, on the other hand, refers to the amount of deformation or change in size that occurs within an object when stress is applied. It measures how much the object changes shape due to the pressure.

So, while stress deals with the external forces acting on an object, strain focuses on the changes happening inside the object because of those forces.


## Load N-ATLaS ASR

In [ ]:
import torch
from transformers import pipeline

ASR_MODELS = {
    "en": "NCAIR1/NigerianAccentedEnglish",
    "ha": "NCAIR1/Hausa-ASR",
    "yo": "NCAIR1/Yoruba-ASR",
    "ig": "NCAIR1/Igbo-ASR",
}

asr_cache = {}


def get_asr(lang):

    if lang not in ASR_MODELS:
        raise ValueError(
            f"Unsupported language: {lang}"
        )

    if lang in asr_cache:
        return asr_cache[lang]

    model_id = ASR_MODELS[lang]

    print(f"\nLoading ASR model: {model_id}")

    asr_pipe = pipeline(
        task="automatic-speech-recognition",
        model=model_id,
        token=HF_TOKEN,
        device=0,
        torch_dtype=torch.float16
    )

    asr_cache[lang] = asr_pipe

    return asr_pipe

Creating ASR function

In [ ]:
def transcribe_audio(
    audio,
    sample_rate,
    lang
):

    asr_pipe = get_asr(lang)

    result = asr_pipe(
        {
            "raw": audio,
            "sampling_rate": sample_rate
        },
        chunk_length_s=30,
        stride_length_s=5
    )

    return result["text"].strip()

Test ASR directly inside colab first

In [ ]:
# import soundfile as sf
#
# audio, sr = sf.read(
#     "/content/ig_01.wav",
#     dtype="float32"
# )
#
# if audio.ndim > 1:
#     audio = audio.mean(axis=1)
#
# print("Sample rate:", sr)
# print("Samples:", len(audio))
#
# text = transcribe_audio(
#     audio,
#     sr,
#     "en"
# )
#
# print("\nTRANSCRIPT:")
# print(text)

## Build FastAPI gateway

In [ ]:
import traceback
import io
import soundfile as sf

from fastapi import (
    FastAPI,
    UploadFile,
    Form,
    Header,
    HTTPException
)

from pydantic import BaseModel

In [ ]:
app = FastAPI(
    title="Muta N-ATLaS Gateway",
    version="0.1.0"
)

In [ ]:
def check_api_key(api_key):

    if api_key != GATEWAY_KEY:

        raise HTTPException(
            status_code=401,
            detail="Invalid API key"
        )

/health

In [ ]:
@app.get("/health")
def health():

    gpu_name = None

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)

    return {
        "ok": True,
        "gpu": gpu_name,
        "asr_loaded": list(
            asr_cache.keys()
        )
    }

### /v1/asr

In [ ]:
@app.post("/v1/asr")
async def asr_endpoint(
    file: UploadFile,
    lang: str = Form(...),
    x_api_key: str = Header(None)
):

    check_api_key(x_api_key)

    if lang not in ASR_MODELS:

        raise HTTPException(
            status_code=400,
            detail=f"Unsupported language: {lang}"
        )

    try:

        raw_audio = await file.read()

        audio, sr = sf.read(
            io.BytesIO(raw_audio),
            dtype="float32"
        )

        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        print(
            f"ASR request: "
            f"lang={lang}, "
            f"sample_rate={sr}, "
            f"samples={len(audio)}"
        )

        if sr != 16000:

            raise ValueError(
                f"Expected 16000 Hz audio, got {sr} Hz"
            )

        text = transcribe_audio(
            audio,
            sr,
            lang
        )

        print(
            f"ASR result: {text}"
        )

        return {
            "text": text,
            "language": lang
        }

    except Exception as exc:

        print("\n========== ASR ERROR ==========")

        traceback.print_exc()

        print(
            "===============================\n"
        )

        raise HTTPException(
            status_code=500,
            detail=str(exc)
        )

/v1/chat

In [ ]:
class ChatRequest(BaseModel):

    messages: list[dict]

    max_tokens: int = 250

    temperature: float = 0.2

In [ ]:
from datetime import datetime


@app.post("/v1/chat")
def chat_endpoint(
    request_data: ChatRequest,
    x_api_key: str = Header(None)
):

    check_api_key(x_api_key)

    prompt = tokenizer.apply_chat_template(
        request_data.messages,
        add_generation_prompt=True,
        tokenize=False,
        date_string=datetime.now().strftime(
            "%d %b %Y"
        )
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    )

    inputs = {
        k: v.to(llm.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=request_data.max_tokens,
            do_sample=(
                request_data.temperature > 0.05
            ),
            temperature=max(
                request_data.temperature,
                0.05
            ),
            top_p=0.9,
            repetition_penalty=1.1
        )

    generated = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    reply = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return {
        "reply": reply
    }

## Start gateway

In [ ]:
import threading
import nest_asyncio
import uvicorn

nest_asyncio.apply()

server_config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(
    server_config
)

threading.Thread(
    target=server.run,
    daemon=True
).start()

print(
    "gateway started."
)

gateway started.


Test gateway inside colab

In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health"
)

print(
    response.json()
)

INFO:     Started server process [998]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:55278 - "GET /health HTTP/1.1" 200 OK
{'ok': True, 'gpu': 'Tesla T4', 'asr_loaded': []}


In [ ]:
import os
import re
import subprocess

subprocess.run(
    """
    wget -q \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O cloudflared
    """,
    shell=True,
    check=True
)

os.chmod(
    "cloudflared",
    0o755
)

In [ ]:
tunnel = subprocess.Popen(
    [
        "./cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

gateway_url = None

while True:

    line = tunnel.stderr.readline()

    if not line:
        break

    print(line.strip())

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        line
    )

    if match:

        gateway_url = match.group(0)

        break

print()
print(
    "GATEWAY_URL =",
    gateway_url
)

2026-10-08T23:11:06Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-08T23:11:06Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-08T23:11:09Z INF +--------------------------------------------------------------------------------------------+
2026-10-08T23:11:09Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-08T23:11:09Z INF |  https://marina-hurricane-modelling-dish.trycloudflare

In [ ]:
import time
import requests
from google.colab import userdata

# 1. Retrieve backend secrets
APP_URL = userdata.get("APP_URL")        # https://muta-rpnz.onrender.com
ADMIN_KEY = userdata.get("ADMIN_KEY")    # Same key stored in Render Environment

def send_heartbeat():
    """Periodically registers the dynamic Cloudflare URL with the Render app."""
    # Correct endpoint path
    register_endpoint = f"{APP_URL.rstrip('/')}/internal/gateway"

    # Correct JSON payload format
    payload = {
        "url": gateway_url
    }

    # Correct header authentication
    headers = {
        "x-admin-key": ADMIN_KEY,
        "Content-Type": "application/json"
    }

    while True:
        try:
            res = requests.post(register_endpoint, json=payload, headers=headers, timeout=10)
            if res.status_code == 200:
                print(f"[{time.strftime('%H:%M:%S')}] Heartbeat success: Registered {gateway_url} with {APP_URL}")
            else:
                print(f"[{time.strftime('%H:%M:%S')}] Heartbeat failed (Status {res.status_code}): {res.text}")
        except Exception as e:
            print(f"[{time.strftime('%H:%M:%S')}] Heartbeat error: {e}")

        time.sleep(60)  # Ping every 60 seconds

# Start heartbeat in background thread
threading.Thread(target=send_heartbeat, daemon=True).start()

In [ ]:
import requests
from google.colab import userdata

app_url = userdata.get("APP_URL")
response = requests.get(f"{app_url}/health")
print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:     74.220.48.219:0 - "GET /health HTTP/1.1" 200 OK
Status Code: 200
Response: {'app': 'ok', 'gateway': {'detail': {'asr_loaded': [], 'gpu': 'Tesla T4', 'ok': True}, 'ok': True}}
